# Chicago Traffic Crashes: Hit and Run Analysis

## Section 1: Setup & Configuration
In this section, we initialize the environment, define global constants, and establish utility functions for consistent visualization styles across the analysis.

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import os
import time
import gc
import contextily as cx
from sklearn.model_selection import train_test_split

# --- Configuration Flags ---
FORCE_DOWNLOAD = False # Set to True to override existence checks and re-download all data
PERFORM_INTEGRATION = True # Set to True to re-run the full join and integration pipeline

# --- API Endpoints ---
CRASHES_API_URL = "https://data.cityofchicago.org/resource/85ca-t3if.json"
VEHICLES_API_URL = "https://data.cityofchicago.org/resource/68nd-jvt3.json"
PEOPLE_API_URL = "https://data.cityofchicago.org/resource/u6pd-qa9d.json"
NIGHTLIFE_API_URL = "https://data.cityofchicago.org/resource/r5kz-chrr.json"

# --- Local Data Paths ---
RAW_CRASHES_FILE = "raw_crashes.parquet"
RAW_VEHICLES_FILE = "raw_vehicles.parquet"
RAW_PEOPLE_FILE = "raw_people.parquet"
NIGHTLIFE_FILE = "nightlife_licenses.parquet"
FINAL_TRAIN_FILE = "train_crashes_final.parquet"
FINAL_TEST_FILE = "test_crashes_final.parquet"

# --- Visualization Settings ---
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
os.makedirs("imgs", exist_ok=True)

def apply_map_style(ax, title):
    """Standardizes map axes and adds a basemap."""
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_axis_off()
    try:
        cx.add_basemap(ax, crs="EPSG:32616", source=cx.providers.CartoDB.Positron)
    except:
        pass

def fetch_socrata_chunked(url, where_clause, chunk_size=50000):
    """
    Robustly fetches a full dataset from Socrata using chunking (offset-based).
    Avoids timeouts and ensures complete data download for large datasets.
    """
    all_data = []
    offset = 0
    while True:
        params = {
            "$limit": chunk_size,
            "$offset": offset,
            "$where": where_clause,
            "$order": ":id" 
        }
        print(f"  Fetching offset {offset}...")
        resp = requests.get(url, params=params)
        data = resp.json()
        if not data or len(data) == 0:
            break
        all_data.extend(data)
        if len(data) < chunk_size:
            break
        offset += chunk_size
        time.sleep(0.5)
    
    return pd.DataFrame(all_data)

## Section 2: Smart Raw Data Acquisition
In this section, we download the necessary datasets. The code checks for the existence of local Parquet files first.

In [2]:
DATE_WHERE = "crash_date >= '2024-01-01T00:00:00' AND crash_date <= '2025-12-31T23:59:59'"

# 1. Fetch Crashes
if not os.path.exists(RAW_CRASHES_FILE) or FORCE_DOWNLOAD:
    print("Downloading Full Crashes (2024-2025)...")
    crashes_where = DATE_WHERE + " AND latitude IS NOT NULL"
    df_crashes_raw = fetch_socrata_chunked(CRASHES_API_URL, crashes_where)
    df_crashes_raw.to_parquet(RAW_CRASHES_FILE)
    print(f"  Saved {len(df_crashes_raw)} raw crashes.")
else:
    print(f"  Local Crashes found. Skipping download.")

# 2. Fetch Vehicles
if not os.path.exists(RAW_VEHICLES_FILE) or FORCE_DOWNLOAD:
    print("\nDownloading Full Vehicles (2024-2025)...")
    df_veh_raw = fetch_socrata_chunked(VEHICLES_API_URL, DATE_WHERE)
    df_veh_raw.to_parquet(RAW_VEHICLES_FILE)
    print(f"  Saved {len(df_veh_raw)} raw vehicles.")
else:
    print(f"  Local Vehicles found. Skipping download.")

# 3. Fetch People
if not os.path.exists(RAW_PEOPLE_FILE) or FORCE_DOWNLOAD:
    print("\nDownloading Full People (2024-2025)...")
    df_peo_raw = fetch_socrata_chunked(PEOPLE_API_URL, DATE_WHERE)
    df_peo_raw.to_parquet(RAW_PEOPLE_FILE)
    print(f"  Saved {len(df_peo_raw)} raw people.")
else:
    print(f"  Local People found. Skipping download.")

# 4. Fetch Nightlife
if not os.path.exists(NIGHTLIFE_FILE) or FORCE_DOWNLOAD:
    print("\nDownloading Nightlife Data...")
    query = "license_status = 'AAI' AND (license_description = 'Tavern' OR license_description = 'Late Hour' OR license_description = 'Consumption on Premises - Incidental Activity') AND latitude IS NOT NULL"
    df_bars_raw = fetch_socrata_chunked(NIGHTLIFE_API_URL, query)
    df_bars_raw.to_parquet(NIGHTLIFE_FILE)
    print(f"  Saved {len(df_bars_raw)} nightlife records.")
else:
    print(f"  Local Nightlife found. Skipping download.")

## Section 3: Full Data Integration Pipeline (Memory-Optimized)
In this section, we process the entire dataset in memory using optimizations to prevent kernel crashes.

In [3]:
if PERFORM_INTEGRATION:
    # --- 1. Load Local Raw Data ---
    print("Loading local raw parquet files...")
    df_c = pd.read_parquet(RAW_CRASHES_FILE)
    df_v = pd.read_parquet(RAW_VEHICLES_FILE)
    df_p = pd.read_parquet(RAW_PEOPLE_FILE)
    gdf_bars = pd.read_parquet(NIGHTLIFE_FILE)
    
    # --- 2. Target Variable & Source Removal ---
    print("Defining Target Variable...")
    # Map 'Y' to 1, everything else to 0. 
    df_c["is_hit_and_run"] = (df_c["hit_and_run_i"].fillna('N') == "Y").astype(int)
    # Drop ONLY the source column once mapped.
    df_c = df_c.drop(columns=["hit_and_run_i"], errors="ignore")
    
    # --- 3. Data Cleaning: Crashes ---
    print("Cleaning and Filtering Crash Data...")
    
    # Note: beat_of_occurrence is RESTORED and excluded from drops.
    LEAKAGE_COLS = [
        'bac_result', 'physical_condition', 'towed_i', 'towed_by', 'fire_i', 
        'hospital', 'ems_agency', 'arrest_related_i', 'statements_taken_i', 
        'photos_taken_i', 'report_type', 'crash_date_est_i', 'lane_cnt', 
        'work_zone_type', 'workers_present_i', 'location', 'date_police_notified'
    ]
    REDUNDANT_COLS = ['street_no', 'street_direction', 'street_name', 'crash_date']
    df_c = df_c.drop(columns=[c for c in LEAKAGE_COLS + REDUNDANT_COLS if c in df_c.columns], errors="ignore")
    
    df_c["latitude"] = pd.to_numeric(df_c["latitude"], errors="coerce")
    df_c["longitude"] = pd.to_numeric(df_c["longitude"], errors="coerce")
    df_c = df_c[(df_c["latitude"] > 41) & (df_c["latitude"] < 43) & (df_c["longitude"] > -88) & (df_c["longitude"] < -87)]

    FLAG_COLS = ['intersection_related_i', 'private_property_i', 'dooring_i', 'work_zone_i']
    for col in FLAG_COLS: df_c[col] = (df_c[col].fillna('N') == "Y").astype(int)
        
    INJURY_COLS = ['injuries_total', 'injuries_fatal', 'injuries_incapacitating', 'injuries_non_incapacitating', 'injuries_reported_not_evident', 'injuries_no_indication', 'injuries_unknown']
    for col in INJURY_COLS: df_c[col] = pd.to_numeric(df_c[col], errors="coerce").fillna(0)
    df_c['most_severe_injury'] = df_c['most_severe_injury'].fillna("NO INDICATION OF INJURY")
    
    # --- 4. Feature Engineering: Vehicles ---
    print("Engineering Behavioral Features...")
    df_v["vehicle_year"] = pd.to_numeric(df_v.get("vehicle_year", 0), errors="coerce")
    df_v.loc[(df_v["vehicle_year"] < 1900) | (df_v["vehicle_year"] > 2026), "vehicle_year"] = np.nan
    
    df_v["truck_involved"] = df_v.get("vehicle_type", pd.Series(dtype=str)).str.contains("TRUCK|TRACTOR", na=False).astype(int)
    df_v["commercial_veh_involved"] = df_v.get("vehicle_use", pd.Series(dtype=str)).str.contains("COMMERCIAL|TAXI|RIDESHARE", na=False).astype(int)
    df_v["public_trans_involved"] = df_v.get("vehicle_use", pd.Series(dtype=str)).str.contains("BUS|CTA", na=False).astype(int)
    df_v["private_use_veh_involved"] = (df_v.get("vehicle_use", pd.Series(dtype=str)) == "PERSONAL").astype(int)
    df_v["unknown_use_veh_involved"] = df_v.get("vehicle_use", pd.Series(dtype=str)).fillna("UNKNOWN").str.contains("UNKNOWN", na=False).astype(int)
    df_v["passenger_type_veh_involved"] = (df_v.get("vehicle_type", pd.Series(dtype=str)) == "PASSENGER").astype(int)
    df_v["unknown_type_veh_involved"] = df_v.get("vehicle_type", pd.Series(dtype=str)).fillna("UNKNOWN").str.contains("UNKNOWN", na=False).astype(int)
    
    v_agg = df_v.groupby("crash_record_id").agg({
        "crash_unit_id": "count", "vehicle_year": "mean", "truck_involved": "max", 
        "commercial_veh_involved": "max", "public_trans_involved": "max", "private_use_veh_involved": "max", 
        "unknown_use_veh_involved": "max", "passenger_type_veh_involved": "max", "unknown_type_veh_involved": "max"
    })
    v_agg.columns = ["total_vehicles_in_crash", "veh_year_avg", "truck_involved", "commercial_veh_involved", "public_trans_involved", "private_use_veh_involved", "unknown_use_veh_involved", "passenger_type_veh_involved", "unknown_type_veh_involved"]
    
    df_p["is_male"] = (df_p.get("sex", pd.Series(dtype=str)) == "M").astype(int)
    df_p["is_female"] = (df_p.get("sex", pd.Series(dtype=str)) == "F").astype(int)
    df_p["airbag_deployed"] = df_p.get("airbag_deployed", pd.Series(dtype=str)).str.contains("DEPLOYED", na=False).astype(int)
    
    p_agg = df_p.groupby("crash_record_id").agg({"is_male": "sum", "is_female": "sum", "airbag_deployed": "max", "person_id": "count"})
    p_agg.columns = ["male_count_in_crash", "female_count_in_crash", "airbag_deployed_in_crash", "total_people_in_crash"]
    
    # --- 5. Optimized Spatial Join ---
    print("Performing Spatial Join (Nightlife Proximity)...")
    gdf_c_temp = gpd.GeoDataFrame(df_c[['longitude', 'latitude']], geometry=gpd.points_from_xy(df_c.longitude, df_c.latitude), crs="EPSG:4326").to_crs("EPSG:32616")
    
    gdf_bars = gdf_bars.drop_duplicates(subset=['latitude', 'longitude'])
    gdf_bars_temp = gpd.GeoDataFrame(gdf_bars[['longitude', 'latitude']], geometry=gpd.points_from_xy(pd.to_numeric(gdf_bars.longitude), pd.to_numeric(gdf_bars.latitude)), crs="EPSG:4326").to_crs("EPSG:32616")
    
    buffered = gdf_c_temp[['geometry']].copy(); buffered["geometry"] = buffered.geometry.buffer(1000)
    joined = gpd.sjoin(buffered, gdf_bars_temp[['geometry']], how="inner", predicate="intersects")
    df_c["nightlife_density_1000m"] = joined.groupby(joined.index).size().reindex(df_c.index, fill_value=0)
    
    del buffered, joined, gdf_c_temp, gdf_bars_temp; gc.collect()
    
    # --- 6. Final Integration & Split ---
    print("Finalizing and Splitting Dataset...")
    df_final = df_c.merge(v_agg, on="crash_record_id", how="left").merge(p_agg, on="crash_record_id", how="left")
    
    agg_cols = ['total_vehicles_in_crash', 'truck_involved', 'commercial_veh_involved', 'public_trans_involved', 'private_use_veh_involved', 'unknown_use_veh_involved', 'passenger_type_veh_involved', 'unknown_type_veh_involved', 'male_count_in_crash', 'female_count_in_crash', 'airbag_deployed_in_crash', 'total_people_in_crash']
    df_final[agg_cols] = df_final[agg_cols].fillna(0)
    
    gdf_final = gpd.GeoDataFrame(df_final, geometry=gpd.points_from_xy(df_final.longitude, df_final.latitude), crs="EPSG:4326")
    df_train, df_test = train_test_split(gdf_final, test_size=0.2, random_state=42, stratify=gdf_final["is_hit_and_run"])
    
    df_train.to_parquet(FINAL_TRAIN_FILE)
    df_test.to_parquet(FINAL_TEST_FILE)
    print(f"Success: Integrated files saved.")

## Section 4: Data Validation & Comprehensive Summary Statistics
In this section, we perform an exhaustive statistical profile and quick spatial visualization of the final integrated dataset.

In [4]:
df_v = gpd.read_parquet(FINAL_TRAIN_FILE)
print(f"Validation: Loaded {len(df_v)} integrated training records.")
print(f"DataFrame Shape: {df_v.shape}")

# 1. Global Null & Duplication Report
print("\n--- Integrity Report ---")
nulls = df_v.isnull().sum()
print(nulls[nulls > 0] if not nulls[nulls > 0].empty else "Zero Missing Values in Feature Matrix!")
print(f"Duplicate Records (by ID): {df_v.duplicated(subset=['crash_record_id']).sum()}")

# 2. Target Variable Balance
print("\n--- Target Distribution (is_hit_and_run) ---")
print(df_v['is_hit_and_run'].value_counts(normalize=True))

# 3. Strategic Behavioral Profile
behavioral_cols = [
    'nightlife_density_1000m', 'unknown_use_veh_involved', 'unknown_type_veh_involved',
    'male_count_in_crash', 'airbag_deployed_in_crash', 'total_vehicles_in_crash'
]
print("\n--- Behavioral Profile: Mean by Crash Type ---")
print(df_v.groupby('is_hit_and_run')[behavioral_cols].mean().T)

# 4. Coordinate Range Validation
print("\n--- Spatial Bounds Check ---")
print(f"Latitude Range:  {df_v.latitude.astype(float).min():.4f} to {df_v.latitude.astype(float).max():.4f}")
print(f"Longitude Range: {df_v.longitude.astype(float).min():.4f} to {df_v.longitude.astype(float).max():.4f}")

# 5. Full Column Listing
print("\n--- All Columns in Dataset ---")
print(df_v.columns.tolist())

# 6. Quick Spatial Heatmap Validation
print("\nGenerating Quick Spatial Validation Heatmap...")
sample_df = df_v.sample(min(10000, len(df_v)), random_state=42).to_crs("EPSG:32616")
hnr_sample = sample_df[sample_df['is_hit_and_run'] == 1]

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

sns.kdeplot(x=hnr_sample.geometry.x, y=hnr_sample.geometry.y, fill=True, cmap="Reds", alpha=0.6, ax=axes[0], thresh=0.05)
apply_map_style(axes[0], "Sample: Hit & Run Density Heatmap")

sc = axes[1].scatter(sample_df.geometry.x, sample_df.geometry.y, c=sample_df['nightlife_density_1000m'], cmap="plasma", s=5, alpha=0.5)
plt.colorbar(sc, ax=axes[1], shrink=0.5, label="Local Nightlife Density (1000m)")
apply_map_style(axes[1], "Sample: Nightlife Proximity Intensity")

plt.tight_layout()
plt.show()